<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **8. Wi-Fi Single Link Throughput**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)

  Simulations may take a few minutes. If running them is not possible, use the saved plots where available.
  
</div>



### 8.1 Introduction

This notebook examines throughput and delay on a Wi-Fi link with one data sender and one receiver. It compares two maximum packet sizes while increasing the offered traffic. Even without competing data senders, headers, acknowledgments, waiting intervals, and backoff consume time.

The course's Wi-Fi MAC material provides background. The first five pages of the following paper introduce IEEE 802.11e channel access:

>S. Mangold, S. Choi, G. R. Hiertz, O. Klein, and B. Walke, [“Analysis of IEEE 802.11e for QoS support in wireless LANs,”](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/802.11/802.11e.pdf) IEEE Wireless Communications, vol. 10, no. 6, pp. 40–50, December 2003.


In [ ]:
import platform

classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")


#### 8.1.1 Clone or update the simulator

Jemula802 is a Java simulator with scenarios configured through XML files. It models traffic, channel access, and packet transmission using simplified radio models. The following cell downloads or updates Jemula802 and its supporting Jemula library. Compilation requires a Java Development Kit (JDK).


In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi

#### 8.1.2 Compile Jemula802 and Jemula


In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

### 8.2 Simulations

Both scenarios use one data sender, one receiver, and the same channel-access settings. They change the maximum generated packet size while keeping the offered bit rates the same. The physical-layer (PHY) rate describes bits transmitted over the radio; delivered throughput also reflects protocol overhead and time spent waiting.


#### 8.2.1 Maximum packet size: 1000 bytes

[MobComp_assignment03_1000byte.xml](jemula802/scenarios/ETH_Course/MobComp_assignment03_1000byte.xml) defines a link on channel 6 with a 54 Mb/s PHY mode. The simulation lasts about 50 s.

At the same sending station, 36 traffic flows start one after another from 2 s to 37 s. Each offers an average of 0.75 Mb/s, so total offered load rises to 27 Mb/s. These are multiple flows sharing one sender, not 36 competing stations.

The `data` traffic generator draws variable packet sizes up to `max_packet_size_byte = 1000` and random interarrival times. Thus, 1000 bytes is a maximum, not a fixed payload size. The configuration sets the fragmentation threshold to 512 bytes and the Request to Send/Clear to Send (RTS/CTS) threshold to 2500 bytes.


In [ ]:
os.chdir(ROOT_DIR)
EXAMPLE_SCENARIO="jemula802/scenarios/ETH_Course/MobComp_assignment03_1000byte.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

#### 8.2.2 Maximum packet size: 120 bytes

[MobComp_assignment03_120byte.xml](jemula802/scenarios/ETH_Course/MobComp_assignment03_120byte.xml) reduces `max_packet_size_byte` to 120. The topology, offered bit rates, and channel-access settings remain the same.

At the same offered bit rate, smaller packets require more packet transmissions. Headers, acknowledgments, and channel-access overhead therefore account for more time per delivered payload bit. The generated packets also fall below the configured fragmentation threshold.

Compare the measured throughput and delay with the 1000-byte case. The effect comes from packet size and protocol overhead; the number of competing data senders does not increase.


In [ ]:
os.chdir(ROOT_DIR)
EXAMPLE_SCENARIO="jemula802/scenarios/ETH_Course/MobComp_assignment03_120byte.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

### 8.3 Evaluation

Results and copies of the scenarios are stored in [results_MobComp_A03_1000byte](results_MobComp_A03_1000byte) and [results_MobComp_A03_120byte](results_MobComp_A03_120byte).

For each scenario, the first plot combines offered load and throughput in Mb/s with average delay in ms. The second shows the delay distribution. Its descending cumulative curve gives the fraction of recorded delays above a threshold, or complementary cumulative distribution function (CCDF); the current legend says CDF.

Compare the scenarios at similar offered loads, paying attention to where throughput stops following the offered traffic.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

os.chdir(ROOT_DIR)

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Avg. Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

#### 8.3.1 Results for the 1000-byte maximum


In [ ]:
if os.path.isdir("results_MobComp_A03_1000byte"):
    os.chdir("results_MobComp_A03_1000byte")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

#### 8.3.2 Results for the 120-byte maximum


In [ ]:
if os.path.isdir("results_MobComp_A03_120byte"):
    os.chdir("results_MobComp_A03_120byte")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

----
### 8.4 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**What does `dot11FragmentationThreshold` control? How does fragmentation change the transmission of a packet larger than this threshold?**

Answer:


**What does `dot11RTSThreshold` control? Under what conditions can an RTS/CTS exchange be useful?**

Answer:


**In section 8.3.1, what is the highest observed throughput? Why is it below the configured 54 Mb/s PHY rate?**

Answer:


**In section 8.3.1, how does average packet delay change as offered load increases? Can propagation time across this short link explain the observed delays?**

Answer:


**What is a packet payload? What are the advantages and disadvantages of larger payloads compared with smaller ones?**

Answer:


**How does throughput differ between the 1000-byte and 120-byte maximum packet sizes at similar offered loads? Explain the difference using protocol overhead.**

Answer:


**How does packet delay differ between the two scenarios at similar offered loads? Explain the result using transmission time, protocol overhead, and queueing.**

Answer:


**OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

Answer:
